In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load the Dataset
df = pd.read_csv("../data/processed/ltv_dataset.csv")
df.head()

#Check shape and info:

df.shape

df.info()

In [ ]:
#Define Features and Target

#Target:

target = "LTV"

#Create X and y:

X = df.drop(columns=["LTV"])
y = df["LTV"]

#Remove customerID because it is only an identifier:

X = X.drop(columns=["customerID"])

In [ ]:
#Define Numerical and Categorical Features

#Use the same feature split as before.

#Numerical features
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen"
]
#Categorical features
categorical_features = [
    col for col in X.columns if col not in numerical_features
]

#Check them:

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

In [ ]:
#Create Train-Test Split

#Use the same split as previous days for fair comparison.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

#Check shapes:

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
#Create Preprocessing Pipelines
#Numeric transformer

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)
#Categorical transformer

categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)
#Combine them

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
#Create the Random Forest Pipeline

#Now create the full ML pipeline with preprocessing + Random Forest model.

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ))
    ]
)

In [ ]:
#Train the Random Forest Regressor

rf_model.fit(X_train, y_train)

In [ ]:
#Make Predictions
#Predictions on training set

y_train_pred = rf_model.predict(X_train)

#Predictions on test set

y_test_pred = rf_model.predict(X_test)

#Preview:

y_test_pred[:10]

In [ ]:
#Evaluate the Random Forest Model
#1) MAE

mae = mean_absolute_error(y_test, y_test_pred)
print("MAE:", mae)

#2) MSE
mse = mean_squared_error(y_test, y_test_pred)
print("MSE:", mse)

#3) RMSE

rmse = np.sqrt(mse)
print("RMSE:", rmse)

#4) R² Score

r2 = r2_score(y_test, y_test_pred)
print("R2 Score:", r2)

In [ ]:
#Create a Results Table
rf_results = pd.DataFrame({
    "Actual_LTV": y_test.values,
    "Predicted_LTV": y_test_pred
})

rf_results.head(10)

In [ ]:
#Plot Actual vs Predicted
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=y_test_pred)
plt.xlabel("Actual LTV")
plt.ylabel("Predicted LTV")
plt.title("Actual vs Predicted LTV - Random Forest")
plt.show()

In [ ]:
#Plot Residual Errors
residuals = y_test - y_test_pred

plt.figure(figsize=(8,6))
sns.histplot(residuals, bins=30, kde=True)
plt.title("Residual Error Distribution - Random Forest")
plt.xlabel("Residual Error")
plt.show()

In [ ]:
#Compare Random Forest vs Linear Regression

#Now create a small comparison table.
#Use the Linear Regression metrics from Day 8 and the new Random Forest metrics.

#Example:

comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [120.5, mae],     # replace 120.5 with your actual Linear Regression MAE
    "MSE": [45000.0, mse],   # replace with actual value
    "RMSE": [212.1, rmse],   # replace with actual value
    "R2 Score": [0.72, r2]   # replace with actual value
})

comparison_df

In [ ]:
#Save the Random Forest Model

#Save the trained model:

joblib.dump(rf_model, "../models/random_forest_ltv.pkl")

#Test loading:

loaded_rf = joblib.load("../models/random_forest_ltv.pkl")
loaded_rf

In [ ]:
### Day 9 Conclusion

# A Random Forest Regressor was trained to predict customer LTV.

# Key observations:
# - Random Forest can capture non-linear relationships better than Linear Regression.
# - Evaluation was performed using MAE, MSE, RMSE, and R² Score.
# - The model can now be compared directly against the Day 7/Day 8 Linear Regression baseline.

# Next step:
# - Compare performance and decide whether Random Forest is better than Linear Regression for LTV prediction.